Comparing the sdf reconstruction precision, given similar amounts of data obtained by TT-SVD and SVD.

- License-Identifier: GPL-3.0-only

- This file is part of the TT-sandbox project.

- Copyright © 2025 Idiap Research Institute <contact@idiap.ch>

- Contributor: Teng Xue <teng.xue@idiap.ch>


In [2]:
import open3d as o3d 
import torch, skimage, sys, os
import numpy as np
import pickle
from utils import tt_svd_full, tt_svd_rank, tt_svd_thres, tt_svd_recon
from utils import svd_full, svd_rank, svd_thres, svd_recon
import plotly.graph_objects as go

sys.path.append(sys.path[0])
device= 'cpu'


%matplotlib inline



In [3]:
def get_mesh(d, level=0.0):
        domain_min = -1
        domain_max = 1
        nbDim = 128
        domain = torch.linspace(domain_min,domain_max,nbDim).to(device)
        domain_half = torch.linspace(domain_min,0,nbDim//2).to(device)
        dms = [domain, domain, domain]
        vw = [nbDim, nbDim, nbDim]

        grid_x, grid_y, grid_z= torch.meshgrid(dms[0], dms[1], dms[2])
        grid_x, grid_y, grid_z = grid_x.reshape(-1,1), grid_y.reshape(-1,1), grid_z.reshape(-1,1)
        p = torch.cat([grid_x, grid_y, grid_z],dim=1).float().to(device)

        # d, _ = self.evaluate(p, var=False, use_derivative=False)

        verts, faces, normals, values = skimage.measure.marching_cubes(
            d.view(vw[0], vw[1], vw[2]).detach().cpu().numpy(), level=level, spacing=np.array([(domain_max-domain_min)/nbDim] * 3)
        )
        verts = verts - [1,1,1]
        return verts, faces, normals

# def mesh_vis(d, cut_x=False, cut_y=False, cut_z=False):
#     # plot the mesh
#     verts, faces, _ = get_mesh(d)
#     rMesh = o3d.geometry.TriangleMesh()
#     rMesh.vertices = o3d.utility.Vector3dVector(verts)
#     rMesh.triangles = o3d.utility.Vector3iVector(faces)
#     rMesh.compute_vertex_normals()
#     mat = o3d.visualization.rendering.MaterialRecord()
#     mat.shader = "defaultLitTransparency"
#     mat.base_color = np.array([1, 1, 1, 1])

#     draw_dicts = [{'name': 'zero', 'geometry': rMesh, 'material': mat}]

#     from open3d.web_visualizer import draw
    
#     o3d.visualization.draw_geometries([draw_dicts[0]['geometry']] )
#     # draw(draw_dicts)

#     # o3d.visualization.draw_geometries(draw_dicts, show_skybox=False)


def mesh_vis(d, cut_x=False, cut_y=False, cut_z=False):
    # Get vertices and faces from mesh
    verts, faces, _ = get_mesh(d)
    
    # Optional: apply axis-aligned cuts
    if cut_x:
        verts = verts[verts[:, 0] >= 0]
    if cut_y:
        verts = verts[verts[:, 1] >= 0]
    if cut_z:
        verts = verts[verts[:, 2] >= 0]

    # Extract x, y, z coordinates and triangle indices
    x, y, z = verts[:, 0], verts[:, 1], verts[:, 2]
    i, j, k = faces[:, 0], faces[:, 1], faces[:, 2]

    # Create a Plotly 3D mesh
    mesh = go.Mesh3d(
        x=x, y=y, z=z,
        i=i, j=j, k=k,
        color='lightblue',
        opacity=1.0,
        flatshading=True,
        name='Mesh'
    )

    layout = go.Layout(
        title='3D Mesh Visualization (Plotly)',
        scene=dict(aspectmode='data'),
        margin=dict(l=0, r=0, t=30, b=0)
    )

    fig = go.Figure(data=[mesh], layout=layout)
    fig.show()


In [4]:
data = np.load(os.getcwd()+"/data/sdf.npy")
d = torch.tensor(data)

mesh_vis(d)

/ssd/anaconda3/envs/tmp/lib/python3.10/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4314.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


## Full TT_SVD

In [5]:
cores = tt_svd_full(data)
data_new = tt_svd_recon(cores).squeeze()

num_elements = 0
for i in range(len(cores)):
    num_elements += cores[i].size
print("The numbers of elements in full tt-svd is:", num_elements)
d = torch.tensor(data_new)
mesh_vis(d)

The ranks of complete 0-th core is (1, 128, 128)
The ranks of complete 1-th core is (128, 128, 128)
The ranks of complete 2-th core is (128, 128, 1)
The numbers of elements in full tt-svd is: 2129920


In [6]:
cores = tt_svd_rank(data, ranks=(50, 50, 50))
data_new = tt_svd_recon(cores).squeeze()
# x_train_new =x_train_recon.reshape(data.shape[0], -1)

num_elements = 0
for i in range(len(cores)):
    num_elements += cores[i].size
error = np.linalg.norm(data - data_new)
print("The l2 norm error is:", error)
print("The numbers of elements in tt-svd is:", num_elements)
d = torch.tensor(data_new)
mesh_vis(d)

The ranks of truncated 0-th core is (1, 128, 50)
The ranks of truncated 1-th core is (50, 128, 50)
The ranks of truncated 2-th core is (50, 128, 1)
The l2 norm error is: 0.2609519
The numbers of elements in tt-svd is: 332800


In [7]:
cores = tt_svd_thres(data, threshold=0.1)
data_new = tt_svd_recon(cores).squeeze()
# x_train_new =x_train_recon.reshape(data.shape[0], -1)

num_elements = 0
for i in range(len(cores)):
    num_elements += cores[i].size
error = np.linalg.norm(data - data_new)
print("The l2 norm error is:", error)
print("The numbers of elements in tt-svd is:", num_elements)
d = torch.tensor(data_new)
mesh_vis(d)

The ranks of truncated 0-th core is (1, 128, 40)
The ranks of truncated 1-th core is (40, 128, 24)
The ranks of truncated 2-th core is (24, 128, 1)
The l2 norm error is: 0.43212333
The numbers of elements in tt-svd is: 131072


## SVD

In [8]:
unfold_tensor = d.reshape(d.shape[0], -1)
print("the shape of unfolding tensor is", unfold_tensor.shape)
U, S, V = svd_full(X=unfold_tensor)
assert V.shape[1]<num_elements, "Number of elements are too few to keep the shape of V matrix. Please increase ranks of TT_SVD!"
r_des = int(num_elements/(U.shape[0]+1+V.shape[1]))+1 #U.shape[0]*r + r + r*V.shape[1] = num_elements
U_new = U[:, :r_des]
S_new = S[:r_des]
V_new = V[:r_des, :]
print(f"U shape: {U_new.shape}, S shape: {S_new.shape}, V shape: {V_new.shape}")
recon_tensor = svd_recon(U_new, S_new, V_new).squeeze()
error = np.linalg.norm(unfold_tensor - recon_tensor)
print("The l2 norm error of standard SVD is:", error)
num_svd_elements = U_new.size + S_new.size + V_new.size
print("The total number of elements in svd is:", num_svd_elements)
d2 = recon_tensor.reshape(128, 128, 128)
mesh_vis(torch.tensor(d2))

the shape of unfolding tensor is torch.Size([128, 16384])
U shape: (128, 8), S shape: (8,), V shape: (8, 16384)
The l2 norm error of standard SVD is: 3.586205
The total number of elements in svd is: 132104
